In [2]:
import torch_geometric.nn as gnn
import torch
import torch.nn as nn
import torch.nn.functional as F

In [12]:
a = torch.randn(2, 3)
b = torch.randn(2, 3)

n = nn.Linear(3, 3)
out_a = n(a)
out_b = n(b)

In [1]:
a = [1, 2, 3]
a[:None]

[1, 2, 3]

In [11]:
a = torch.randint(2, 5, (2, 5))
print(a)

print(torch.min(a, dim=-1))

tensor([[4, 3, 2, 4, 4],
        [3, 2, 2, 4, 2]])
torch.return_types.min(
values=tensor([2, 2]),
indices=tensor([2, 1]))


In [2]:
import torch

cost = torch.randn(2, 5)
pop = torch.randn(2, 5, 3)
gbest_val = torch.randn(2)
gbest = torch.randn(2, 3)
print(f"cost: {cost}")
print(f"pop: {pop}")
print(f"gbest_val: {gbest_val}")
print(f"gbest: {gbest}")

min_cost, min_idx = torch.min(cost, dim=1) # shape (2,), shape (2,)
print(f"min_cost: {min_cost}")
print(f"min_idx: {min_idx}")
better_gbest_mask = min_cost < gbest_val # shape (2,)
print(f"better_gbest_mask: {better_gbest_mask}")
gbest_val[better_gbest_mask] = min_cost[better_gbest_mask]
gbest[better_gbest_mask] = pop[better_gbest_mask, min_idx[better_gbest_mask]].detach().clone() # shape (2, 3)
print(f"updated gbest_val: {gbest_val}")
print(f"updated gbest: {gbest}")



cost: tensor([[ 1.0686, -0.8237, -1.1987,  0.2698,  0.2072],
        [-0.3634, -1.0371,  0.6659,  0.8174,  0.0032]])
pop: tensor([[[ 1.3943, -0.7776,  0.4244],
         [-1.2440, -0.7131,  0.4648],
         [ 0.0236, -0.5719, -3.1174],
         [ 1.2810,  1.1222, -0.6873],
         [ 0.4607, -1.6635, -0.4395]],

        [[-0.4652, -0.2316,  0.4585],
         [-1.5650,  0.2545, -0.3222],
         [ 0.8499,  0.8087, -0.3978],
         [-0.4876,  0.9476, -0.1732],
         [-0.1778,  0.7817,  0.2898]]])
gbest_val: tensor([-0.5252, -0.8842])
gbest: tensor([[ 0.2639, -0.3230, -1.1047],
        [ 0.5717, -1.8684, -0.5462]])
min_cost: tensor([-1.1987, -1.0371])
min_idx: tensor([2, 1])
better_gbest_mask: tensor([True, True])
updated gbest_val: tensor([-1.1987, -1.0371])
updated gbest: tensor([[ 0.0236, -0.5719, -3.1174],
        [-1.5650,  0.2545, -0.3222]])


In [6]:
gbest.shape

torch.Size([2, 3])

In [8]:
from torch_geometric.data import Batch, Data


problems = [
    Data(x=torch.randn(5, 3), edge_index=torch.tensor([[0, 1, 2, 3], [1, 0, 3, 2]]), edge_attr=torch.randn(4, 2)),
    Data(x=torch.randn(5, 3), edge_index=torch.tensor([[0, 1, 2, 4], [1, 2, 0, 4]]), edge_attr=torch.randn(4, 2))
]
batch = Batch.from_data_list(problems)

In [14]:
print(batch.edge_index)
print(batch.edge_attr.shape)
print(batch.x.shape)

tensor([[0, 1, 2, 3, 5, 6, 7, 9],
        [1, 0, 3, 2, 6, 7, 5, 9]])
torch.Size([8, 2])
torch.Size([10, 3])


In [15]:
batch_size =2
n_cities = 5
n_dims = 2
device = 'cpu'
coordinates = torch.rand(batch_size, n_cities, n_dims, device=device)
distance_matrix = torch.norm(coordinates[:, :, None] - coordinates[:, None, :], dim=3, p=2)
distance_matrix[torch.arange(batch_size)[:, None], torch.arange(n_cities), torch.arange(n_cities)] = 1e9  # Prevent zero distance to self


In [16]:
distance_matrix

tensor([[[1.0000e+09, 5.5601e-01, 2.3655e-01, 1.9464e-01, 2.5040e-01],
         [5.5601e-01, 1.0000e+09, 6.8004e-01, 7.4263e-01, 7.0828e-01],
         [2.3655e-01, 6.8004e-01, 1.0000e+09, 1.8323e-01, 2.9237e-02],
         [1.9464e-01, 7.4263e-01, 1.8323e-01, 1.0000e+09, 1.7135e-01],
         [2.5040e-01, 7.0828e-01, 2.9237e-02, 1.7135e-01, 1.0000e+09]],

        [[1.0000e+09, 2.3016e-01, 4.2855e-01, 6.1370e-01, 2.8450e-01],
         [2.3016e-01, 1.0000e+09, 3.8241e-01, 5.3223e-01, 2.9551e-01],
         [4.2855e-01, 3.8241e-01, 1.0000e+09, 9.1398e-01, 1.4928e-01],
         [6.1370e-01, 5.3223e-01, 9.1398e-01, 1.0000e+09, 8.1999e-01],
         [2.8450e-01, 2.9551e-01, 1.4928e-01, 8.1999e-01, 1.0000e+09]]])

In [29]:
def for_loop_batch():
    k_sparse = 2
    batch_size, n_cities, _ = coordinates.shape
    topk_values, topk_indices = torch.topk(distance_matrix, 
                                        k=k_sparse, 
                                        dim=2, largest=False)
    probs = []
    for i in range(batch_size):
        edge_index = torch.stack([
            torch.repeat_interleave(torch.arange(n_cities).to(topk_indices[i].device),
                                    repeats=k_sparse),
            torch.flatten(topk_indices[i])
            ])
        probs.append(Data(x=coordinates[i], edge_index=edge_index, edge_attr=topk_values[i].reshape(-1, 1)))
    batch = Batch.from_data_list(probs)
    return batch

In [30]:
%timeit for_loop_batch()

569 μs ± 26.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
def vectorized_batch():
    k_sparse = 2
    batch_size, n_cities, _ = coordinates.shape
    topk_values, topk_indices = torch.topk(distance_matrix, 
                                        k=k_sparse, 
                                        dim=2, largest=False)

    # Vectorized edge_index construction
    # Source nodes: repeat [0,1,...,n_cities-1] k_sparse times for each batch
    src = torch.arange(n_cities, device=coordinates.device).unsqueeze(1).expand(-1, k_sparse).reshape(-1)
    src = src.unsqueeze(0).expand(batch_size, -1)  # (batch_size, n_cities * k_sparse)
    # Destination nodes: flatten topk_indices
    dst = topk_indices.reshape(batch_size, -1)  # (batch_size, n_cities * k_sparse)

    # Add batch offsets to create a single batched graph
    batch_offsets = torch.arange(batch_size, device=coordinates.device) * n_cities
    src_batched = (src + batch_offsets[:, None]).reshape(-1)
    dst_batched = (dst + batch_offsets[:, None]).reshape(-1)
    edge_index = torch.stack([src_batched, dst_batched])
    # Flatten node features and edge attributes
    x_batched = coordinates.reshape(-1, coordinates.size(-1))  # (batch_size * n_cities, n_dims)
    edge_attr_batched = topk_values.reshape(-1, 1)  # (batch_size * n_cities * k_sparse, 1)

    # Create batch assignment tensor for PyG
    batch_tensor = torch.arange(batch_size, device=coordinates.device).repeat_interleave(n_cities)

    # Create a single Batch object directly
    batch = Batch(x=x_batched, edge_index=edge_index, edge_attr=edge_attr_batched, batch=batch_tensor)
    return batch
vectorized_batch()

tensor([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9])


DataBatch(x=[10, 2], edge_index=[2, 20], edge_attr=[20, 1], batch=[10])

In [35]:
distance_matrix = torch.randint(1, 10, (2, 5, 5))
distance_matrix[torch.arange(2)[:, None], torch.arange(5), torch.arange(5)] = 1e9  # Prevent zero distance to self
print(distance_matrix)

tensor([[[1000000000,          8,          8,          1,          6],
         [         4, 1000000000,          1,          1,          8],
         [         2,          6, 1000000000,          8,          5],
         [         6,          1,          5, 1000000000,          1],
         [         7,          4,          9,          1, 1000000000]],

        [[1000000000,          1,          7,          3,          2],
         [         3, 1000000000,          1,          1,          9],
         [         1,          4, 1000000000,          4,          3],
         [         8,          5,          1, 1000000000,          9],
         [         5,          9,          2,          6, 1000000000]]])


In [42]:
solutions = torch.rand(2, 2, 5).argsort(dim=-1)
solutions

tensor([[[2, 1, 0, 4, 3],
         [3, 4, 0, 1, 2]],

        [[2, 0, 1, 4, 3],
         [0, 4, 2, 1, 3]]])

In [53]:
from functools import cached_property


class ABCD:
    def __init__(self):
        self.x = 10

    @cached_property
    def y(self):
        return self.x * 2
    
    @classmethod
    def from_x(cls, x):
        instance = cls()
        instance.x = x
        return instance
a = ABCD()
print(a.y)

20


In [ ]:
starts = torch.rand(
    batch_size, 3, 5, device="cpu"
).argsort(dim=-1)[..., :2]


# Expand population: (batch_size, n_particles, dim) -> (batch_size, n_particles * n_starts, dim)
expanded_population = original_population.unsqueeze(2).expand(
    -1, -1, n_starts, -1
)

30
30


In [63]:
import torch

edge_index = torch.tensor([[0, 1, 2], [1, 2, 0]])

dim = torch.rand(2, 3).unsqueeze(1).expand(-1, 2, -1).reshape(-1, 3)
print(f"dim: {dim}")

mat = torch.ones((4, 3, 3)) * (-1)

mat[:, edge_index[0], edge_index[1]] = dim
mat

dim: tensor([[0.0703, 0.6673, 0.3153],
        [0.0703, 0.6673, 0.3153],
        [0.9150, 0.8258, 0.9986],
        [0.9150, 0.8258, 0.9986]])


tensor([[[-1.0000,  0.0703, -1.0000],
         [-1.0000, -1.0000,  0.6673],
         [ 0.3153, -1.0000, -1.0000]],

        [[-1.0000,  0.0703, -1.0000],
         [-1.0000, -1.0000,  0.6673],
         [ 0.3153, -1.0000, -1.0000]],

        [[-1.0000,  0.9150, -1.0000],
         [-1.0000, -1.0000,  0.8258],
         [ 0.9986, -1.0000, -1.0000]],

        [[-1.0000,  0.9150, -1.0000],
         [-1.0000, -1.0000,  0.8258],
         [ 0.9986, -1.0000, -1.0000]]])

In [43]:
batch_size, n_particles, n_cities = solutions.shape
u = solutions  # shape: (batch_size, n_particles, n_cities)
v = torch.roll(u, shifts=-1, dims=2)  # shape: (batch_size, n_particles, n_cities)
batch_indices = torch.arange(batch_size, device=solutions.device).unsqueeze(1).unsqueeze(2).expand(-1, n_particles, -1)
assert (distance_matrix[batch_indices, u, v] > 0).all()
print(torch.sum(distance_matrix[batch_indices, u, v], dim=2))

tensor([[22, 25],
        [18, 17]])


In [2]:
import pytorch_lightning as L

trainer = L.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices="auto",
    num_sanity_val_steps=0,
    log_every_n_steps=10,
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [10]:
import yaml
d = yaml.safe_load(open("configs/aco_tsp.yaml"))
print(d)

{'env': {'problem_cls': 'TSPProblem', 'n_particles': 128, 'step_per_epoch': 128, 'n_cities': 20, 'k_sparse': 5}, 'aco_agent': {'n_cities': 20, 'n_ants': 20, 'mode': 'choice', 'aco_iterations_infer': 20}, 'trainer': {'max_epochs': 5, 'accelerator': 'auto', 'devices': 'auto', 'num_sanity_val_steps': 0, 'log_every_n_steps': 16}}


In [ ]:
d = {
    "a": [1, 2, 3],
    "b": [4, 5, 6],
}
d_plus1 = {
    k: [x + 1 for x in v] for k, v in d.items()
}

'cuda'

In [4]:
from torch.distributions import Normal
mean = torch.zeros(5, 5, 3)
std = torch.ones(5, 5, 3)
dist = Normal(mean, std)
samples = dist.sample()
print(samples.shape)
print(dist.log_prob(samples).shape)
print(dist.entropy().shape)

torch.Size([5, 5, 3])
torch.Size([5, 5, 3])
torch.Size([5, 5, 3])


In [3]:
from torch.distributions import Categorical

t = torch.tensor([[0.1, float('-inf'), 0.7], [2.0, 1.0, 0.1]])
d = Categorical(logits=t)

In [4]:
t = torch.tensor([0.0, 0.0, 0.0])
d = Categorical(t)

ValueError: Expected parameter probs (Tensor of shape (3,)) of distribution Categorical(probs: torch.Size([3])) to satisfy the constraint Simplex(), but found invalid values:
tensor([nan, nan, nan])

In [10]:
mat = torch.zeros((n_particles, n_cities, n_cities))
mat[edge_index[0], edge_index[1]] = edge_values

RuntimeError: shape mismatch: value tensor of shape [3, 15] cannot be broadcast to indexing result of shape [15, 5]

In [9]:
edge_values

tensor([[0.3569, 0.4387, 0.8685, 0.9301, 0.3460, 0.7346, 0.7125, 0.7116, 0.0034,
         0.8694, 0.1189, 0.6987, 0.4217, 0.2760, 0.2783],
        [0.1926, 0.8375, 0.0232, 0.6832, 0.7921, 0.0502, 0.7298, 0.2785, 0.8785,
         0.4541, 0.8964, 0.0214, 0.7403, 0.7855, 0.6148],
        [0.5817, 0.5351, 0.1427, 0.6011, 0.6688, 0.9171, 0.1967, 0.2369, 0.4065,
         0.1424, 0.7792, 0.4304, 0.4078, 0.5154, 0.2953]])

In [8]:
mat

tensor([[[0.0000, 0.3569, 0.4387, 0.8685, 0.0000],
         [0.9301, 0.0000, 0.3460, 0.0000, 0.7346],
         [0.7125, 0.7116, 0.0000, 0.0000, 0.0034],
         [0.8694, 0.0000, 0.1189, 0.0000, 0.6987],
         [0.0000, 0.4217, 0.2760, 0.2783, 0.0000]],

        [[0.0000, 0.1926, 0.8375, 0.0232, 0.0000],
         [0.6832, 0.0000, 0.7921, 0.0000, 0.0502],
         [0.7298, 0.2785, 0.0000, 0.0000, 0.8785],
         [0.4541, 0.0000, 0.8964, 0.0000, 0.0214],
         [0.0000, 0.7403, 0.7855, 0.6148, 0.0000]],

        [[0.0000, 0.5817, 0.5351, 0.1427, 0.0000],
         [0.6011, 0.0000, 0.6688, 0.0000, 0.9171],
         [0.1967, 0.2369, 0.0000, 0.0000, 0.4065],
         [0.1424, 0.0000, 0.7792, 0.0000, 0.4304],
         [0.0000, 0.4078, 0.5154, 0.2953, 0.0000]]])

In [4]:
t = torch.randn(4, 16)  # Example input tensor with shape (4, 16)
o = F.gumbel_softmax(t, tau=1, hard=True)
print(o)

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


In [ ]:
def decode_tour_differentiable(particle_X, temperature=1.0):
    num_cities = particle_X.shape
    tour = []
    mask = torch.zeros(num_cities)
    
    for _ in range(num_cities):
        # Apply mask to priorities
        masked_logits = particle_X.masked_fill(mask.bool(), float('-inf'))
        
        # Gumbel-Softmax for differentiable sampling
        # Use hard=True for a one-hot vector in forward, softmax in backward
        prob_vec = torch.nn.functional.gumbel_softmax(masked_logits, tau=temperature, hard=True)
        
        city_idx = prob_vec.argmax()
        tour.append(city_idx)
        mask[city_idx] = 1
        
    return tour # Used for cost evaluation

In [4]:
conv = gnn.GCNConv(16, 32)
i = torch.randn(4, 16)
edge_index = torch.tensor([[0, 1, 2, 3, 0, 2],
                           [1, 0, 3, 2, 2, 0]], dtype=torch.long)
o = conv(i, edge_index)
print(o.shape)

torch.Size([4, 32])


In [9]:
o = nn.AdaptiveAvgPool1d(1)(torch.randn(4, 16, 10))  # IGNORE
print(o.shape)  # IGNORE

torch.Size([4, 16, 1])


In [22]:
gnn.knn_graph(torch.randn(5, 3), k=6)  # IGNORE

tensor([[2, 3, 4, 1, 4, 0, 2, 3, 0, 3, 4, 1, 2, 0, 4, 1, 1, 2, 0, 3],
        [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4]])

In [27]:
n_cities = 5
if False:
    row = torch.arange(n_cities).unsqueeze(1).repeat(1, n_cities).view(-1)
    col = torch.arange(n_cities).unsqueeze(0).repeat(n_cities, 1).view(-1)
else:
    row = torch.arange(n_cities).unsqueeze(1).repeat(1, n_cities - 1).view(-1)
    col = torch.cat([torch.cat([torch.arange(i), torch.arange(i + 1, n_cities)]) for i in range(n_cities)], dim=0)
edge_index = torch.stack([row, col], dim=0)
edge_index

tensor([[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4],
        [1, 2, 3, 4, 0, 2, 3, 4, 0, 1, 3, 4, 0, 1, 2, 4, 0, 1, 2, 3]])

In [31]:
gnn.knn_graph(torch.randn(n_cities, 3), k=n_cities, loop=False)  # IGNORE

tensor([[3, 4, 2, 1, 4, 2, 0, 3, 0, 4, 3, 1, 0, 4, 2, 1, 3, 0, 2, 1],
        [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4]])

In [5]:
from problems import ProblemDataModule, TSPProblem

In [6]:
data_module = ProblemDataModule(
    problem_cls=TSPProblem,
    step_per_epoch=128,
    n_cities=50,
    k_sparse=10,
    n_dims=2,    
)

In [22]:
import torch

n_particles = 20
n_starts = 5
n_cities = 5

starts = []
for _ in range(n_particles):
    starts.append(torch.randperm(n_cities)[:n_starts])
torch.stack(starts, dim=0)

tensor([[3, 0, 1, 4, 2],
        [4, 0, 3, 2, 1],
        [3, 1, 0, 2, 4],
        [0, 3, 1, 4, 2],
        [4, 1, 3, 0, 2],
        [3, 2, 0, 1, 4],
        [4, 3, 2, 0, 1],
        [2, 4, 0, 1, 3],
        [4, 1, 0, 3, 2],
        [2, 3, 4, 0, 1],
        [3, 0, 4, 1, 2],
        [4, 2, 1, 3, 0],
        [2, 1, 3, 4, 0],
        [0, 3, 2, 4, 1],
        [0, 3, 4, 1, 2],
        [1, 3, 0, 2, 4],
        [1, 4, 0, 3, 2],
        [4, 1, 2, 3, 0],
        [0, 3, 1, 4, 2],
        [3, 0, 1, 4, 2]])

In [25]:
# Vectorized alternative to generate starts (no Python loop)
starts = torch.rand(n_particles, n_cities).argsort(dim=1)[:, :1]
starts.T

tensor([[4, 3, 2, 3, 3, 2, 2, 2, 2, 1, 4, 4, 1, 1, 4, 3, 2, 1, 0, 3]])

In [7]:
train_dataloader = data_module.train_dataloader()
cnt = 0
for problem in train_dataloader:
    cnt += 1

In [8]:
problem.pyg_data

Data(x=[50, 2], edge_index=[2, 500], edge_attr=[500, 1])

In [8]:
problem.pyg_data = problem.pyg_data.to("cuda")

In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from copy import deepcopy
import torch_geometric.nn as gnn

# GNN for edge embeddings
class EmbNet(nn.Module):
    def __init__(self, depth=12, feats=2, units=32, act_fn='silu', agg_fn='mean'): # TODO feats=1
        super().__init__()
        self.depth = depth
        self.feats = feats
        self.units = units
        self.act_fn = getattr(F, act_fn)
        self.agg_fn = getattr(gnn, f'global_{agg_fn}_pool')
        self.v_lin0 = nn.Linear(self.feats, self.units)
        self.v_lins1 = nn.ModuleList([nn.Linear(self.units, self.units) for i in range(self.depth)])
        self.v_lins2 = nn.ModuleList([nn.Linear(self.units, self.units) for i in range(self.depth)])
        self.v_lins3 = nn.ModuleList([nn.Linear(self.units, self.units) for i in range(self.depth)])
        self.v_lins4 = nn.ModuleList([nn.Linear(self.units, self.units) for i in range(self.depth)])
        self.v_bns = nn.ModuleList([gnn.BatchNorm(self.units) for i in range(self.depth)])
        self.e_lin0 = nn.Linear(1, self.units)
        self.e_lins0 = nn.ModuleList([nn.Linear(self.units, self.units) for i in range(self.depth)])
        self.e_bns = nn.ModuleList([gnn.BatchNorm(self.units) for i in range(self.depth)])
    def reset_parameters(self):
        raise NotImplementedError
    def forward(self, x, edge_index, edge_attr):
        x = x
        w = edge_attr
        x = self.v_lin0(x)
        x = self.act_fn(x)
        w = self.e_lin0(w)
        w = self.act_fn(w)
        for i in range(self.depth):
            x0 = x
            x1 = self.v_lins1[i](x0)
            x2 = self.v_lins2[i](x0)
            x3 = self.v_lins3[i](x0)
            x4 = self.v_lins4[i](x0)
            w0 = w
            w1 = self.e_lins0[i](w0)
            w2 = torch.sigmoid(w0)
            x = x0 + self.act_fn(self.v_bns[i](x1 + self.agg_fn(w2 * x2[edge_index[1]], edge_index[0])))
            w = w0 + self.act_fn(self.e_bns[i](w1 + x3[edge_index[0]] + x4[edge_index[1]]))
        return x

In [ ]:
import torch_geometric.nn as gnn
import torch

gcn_net = gnn.GCN(in_channels=2, hidden_channels=16, num_layers=2, out_channels=32)


torch.Size([6, 5, 32])

In [9]:
x = torch.randn(6, 5, 2)
x1 = x[0]
x4 = x[4]
# Define a full edge index for a graph with 5 nodes
edge_index = torch.tensor([
    [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4],
    [1, 2, 3, 4, 0, 2, 3, 4, 0, 1, 3, 4, 0, 1, 2, 4, 0, 1, 2, 3]
], dtype=torch.long)
out = gcn_net(x, edge_index)
print(out.shape)
out1 = gcn_net(x1, edge_index)
out4 = gcn_net(x4, edge_index)
print(out1.shape)
print(out4.shape)
print((out[0] - out1).abs().max())
print((out[4] - out4).abs().max())

torch.Size([6, 5, 32])
torch.Size([5, 32])
torch.Size([5, 32])
tensor(0., grad_fn=<MaxBackward1>)
tensor(0., grad_fn=<MaxBackward1>)


In [10]:
from torch_geometric.utils import add_self_loops, degree

edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
edge_index

tensor([[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 0, 1, 2, 3,
         4, 5],
        [1, 2, 3, 4, 0, 2, 3, 4, 0, 1, 3, 4, 0, 1, 2, 4, 0, 1, 2, 3, 0, 1, 2, 3,
         4, 5]])

In [ ]:
net = EmbNet(feats=2)
import torch

x = torch.randn(5, 2)
edge_index = torch.tensor([[0, 1, 2, 3, 4, 0],
                           [1, 0, 3, 2, 0, 4]], dtype=torch.long)

x, edge_index, edge_attr = problem.pyg_data.x, problem.pyg_data.edge_index, problem.pyg_data.edge_attr

out = net(x, edge_index, edge_attr)
out.shape

torch.Size([50, 32])

In [12]:
out.T.shape

torch.Size([32, 50])

In [13]:
import torch.distributions as dst
import torch

t = torch.rand(5)
print(t)

dist = dst.Categorical(logits=t)
for _ in range(10):
    sample = dist.sample()
    print(sample, dist.log_prob(sample))

tensor([0.3209, 0.2086, 0.4127, 0.1192, 0.5564])
tensor(0) tensor(-1.6239)
tensor(4) tensor(-1.3884)
tensor(4) tensor(-1.3884)
tensor(4) tensor(-1.3884)
tensor(0) tensor(-1.6239)
tensor(0) tensor(-1.6239)
tensor(2) tensor(-1.5321)
tensor(0) tensor(-1.6239)
tensor(4) tensor(-1.3884)
tensor(4) tensor(-1.3884)


In [1]:
import torch.distributions as dst
import torch
t = torch.rand(5)
print(t)
t5 = t*5
print(t5)
dt = dst.Categorical(logits=t)
dt5 = dst.Categorical(logits=t5)

tensor([0.3874, 0.5897, 0.0379, 0.4437, 0.1253])
tensor([1.9370, 2.9487, 0.1896, 2.2185, 0.6266])


In [9]:
n = 20
d = dst.Categorical(probs=torch.ones(n)/n)
for i in range(10):
    sample = d.sample()
    print(sample, d.log_prob(sample))

tensor(18) tensor(-2.9957)
tensor(1) tensor(-2.9957)
tensor(0) tensor(-2.9957)
tensor(19) tensor(-2.9957)
tensor(16) tensor(-2.9957)
tensor(19) tensor(-2.9957)
tensor(13) tensor(-2.9957)
tensor(0) tensor(-2.9957)
tensor(13) tensor(-2.9957)
tensor(8) tensor(-2.9957)


In [34]:
def sample_solution(tensor: torch.Tensor) -> torch.Tensor:
    n = tensor.size(0)
    seq = []
    for _ in range(n):
        tensor = tensor / tensor.sum()
        d = dst.Categorical(probs=tensor)
        sample = d.sample()
        seq.append(sample.item())
        # Mask the selected city
        tensor[sample] = 0.0
    return torch.tensor(seq, device=tensor.device)


In [41]:
mat = torch.rand(10, 5)
d = dst.Categorical(probs=mat)
d0 = d.sample()
d1 = d.sample()

torch.stack([d0, d1]).shape

torch.Size([2, 10])

In [39]:
for i in range(10):
    print(sample_solution(torch.ones(5)/5))
    print("---")

tensor([1, 4, 0, 3, 2])
---
tensor([0, 4, 2, 3, 1])
---
tensor([0, 4, 3, 1, 2])
---
tensor([4, 1, 2, 0, 3])
---
tensor([2, 3, 4, 0, 1])
---
tensor([0, 4, 1, 2, 3])
---
tensor([4, 3, 2, 1, 0])
---
tensor([3, 0, 2, 4, 1])
---
tensor([4, 0, 2, 1, 3])
---
tensor([1, 4, 0, 3, 2])
---


In [26]:
dt.logits

tensor([-2.0553, -1.5910, -1.1634, -1.9179, -1.5660])

In [28]:
softmax = torch.nn.Softmax(dim=0)
print(softmax(t))
print(softmax(t5))

tensor([0.1281, 0.2037, 0.3124, 0.1469, 0.2089])
tensor([0.0090, 0.0916, 0.7776, 0.0179, 0.1039])


In [18]:
for i in range(10):
    sample = dt.sample()
    sample5 = dt5.sample()
    print(sample, dt.log_prob(sample), sample5, dt5.log_prob(sample5))

tensor(3) tensor(-1.5452) tensor(3) tensor(-1.7250)
tensor(0) tensor(-1.3677) tensor(0) tensor(-0.8379)
tensor(3) tensor(-1.5452) tensor(3) tensor(-1.7250)
tensor(4) tensor(-1.4190) tensor(4) tensor(-1.0942)
tensor(1) tensor(-2.0031) tensor(0) tensor(-0.8379)
tensor(0) tensor(-1.3677) tensor(3) tensor(-1.7250)
tensor(0) tensor(-1.3677) tensor(0) tensor(-0.8379)
tensor(0) tensor(-1.3677) tensor(4) tensor(-1.0942)
tensor(4) tensor(-1.4190) tensor(0) tensor(-0.8379)
tensor(0) tensor(-1.3677) tensor(4) tensor(-1.0942)


In [36]:
import torch

N = torch.randint(20, 100, (1,)).item()
M = torch.randint(20, 500, (1,)).item()

t = torch.randn(1, 1, N, M)
print(t.shape)

import torch.nn as nn
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = torch.nn.functional.silu(x)
        return x
modules = nn.ModuleList()
init_block = ConvBlock(in_channels=1, out_channels=3, kernel_size=(3,3), padding="same")
for i in range(5):
    print(in_channels:=3+max(i-1, 0)*2, out_channels:=3+i*2)
    modules.append(ConvBlock(in_channels=3+max(i-1, 0)*2, out_channels=3+i*2, kernel_size=(3+i*2,3+i*2), padding="same"))

torch.Size([1, 1, 76, 369])
3 3
3 5
5 7
7 9
9 11


In [37]:
o = init_block(t)
for module in modules:
    o = module(o)
o.shape

torch.Size([1, 11, 76, 369])

In [42]:
nn.Softmax(dim=-1)(o).shape

torch.Size([1, 11, 76, 369])

In [2]:
import torch
import torch.nn as nn

in_channels = 128
out_channels = 64
n = nn.Conv1d(in_channels, out_channels, 3, padding="same")
t = torch.randn(1, in_channels, 50)

o = n(t)
o.shape

torch.Size([1, 64, 50])

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding, act_fn='silu'):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act_fn = getattr(F, act_fn)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.act_fn(x)
        return x

class TSPVectorCNNModel(nn.Module):
    def __init__(self, hidden_channels=16, depth=5):
        super().__init__()
        self.init_block = ConvBlock(1, hidden_channels, kernel_size=3, padding="same")
        self.conv_modules = nn.ModuleList()
        for i in range(depth):
            in_channels = hidden_channels if i == 0 else hidden_channels + (i - 1) * 2
            out_channels = hidden_channels + i * 2
            self.conv_modules.append(ConvBlock(in_channels, out_channels, kernel_size=3, padding="same"))
        self.lin = nn.Linear(1, 3)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Args:
            x: torch tensor with shape (n_particles, n_cities)
        Returns:
            logits: torch tensor with shape (n_particles, 3), logits for (w, c1, c2)
        '''
        x = x.view(1, 1, x.size(0), x.size(1))  
        o = self.init_block(x)
        for module in self.conv_modules:
            o = module(o)
        o = o.mean(dim=[1, 3], keepdim=True)  # shape: (1, 1, n_particles, 1)
        print(o.shape)
        print(o.squeeze(dim=-1).squeeze(dim=0).shape)
        o = self.lin(o).squeeze()  # shape: (n_particles, 3)
        return self.softmax(o)

In [55]:
net = TSPVectorCNNModel()
net(torch.rand(10, 20)).shape

torch.Size([1, 1, 10, 1])
torch.Size([1, 10])


torch.Size([10, 3])

In [1]:
from nets.focalsvtr import FocalSVTR
import torch

m = FocalSVTR(embed_dim=16)
i = torch.randn(1, 3, 128, 8)

o = m(i)
o.shape


Initializing FocalNetBlock with dim=16, input_resolution=[8, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=16, input_resolution=[8, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=16, input_resolution=[8, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=32, input_resolution=[4, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=32, input_resolution=[4, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=32, input_resolution=[4, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=32, input_resolution=[4, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=32, input_resolution=[4, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=32, input_resolution=[4, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=64, input_resolution=[2, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=64, input_resolution=[2, 32], mlp_ratio=4.0
Initializing FocalNetBlock with dim=64, input_resolution=[2, 32], mlp_ratio=4.0
x shape after patch embed: torch.Size([1

RuntimeError: Given normalized_shape=[32], expected input with shape [*, 32], but got input of size[1, 64, 16]

In [3]:
import torch.nn as nn
import torch

m = nn.Conv2d(3, 16, kernel_size=[2, 1], stride=[2, 1])
i = torch.randn(1, 3, 128, 8)
o = m(i)
o.shape

torch.Size([1, 16, 64, 8])

In [4]:
m = nn.Conv1d(3, 16, kernel_size=3, stride=1, padding=1)
i = torch.randn(1, 3, 128)
o = m(i)
o.shape

torch.Size([1, 16, 128])

In [7]:
nn.AvgPool1d(kernel_size=2)(o).shape

torch.Size([1, 16, 64])

In [15]:
class Conv1dBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding, act_fn='silu'):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act_fn = getattr(F, act_fn)
    def forward(self, x, last=False):
        x = self.conv(x)
        x = self.bn(x)
        if not last:
            x = self.act_fn(x)
        return x

class TSPVectorCNN1DModel(nn.Module):
    def __init__(self, n_particles=128, emb_dim=32, act_fn='silu'):
        super().__init__()
        self.n_particles = n_particles
        self.avgpool = nn.AvgPool1d(kernel_size=2)
        self.conv1 = Conv1dBlock(n_particles, n_particles//2, kernel_size=3, padding=1, act_fn=act_fn)
        self.conv2 = Conv1dBlock(n_particles//2, n_particles*2, kernel_size=3, padding=1, act_fn=act_fn)
        self.conv3 = Conv1dBlock(n_particles*2, n_particles, kernel_size=3, padding=1, act_fn=act_fn)
        self.avg_global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(1, emb_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Args:
            x: torch tensor with shape (n_particles, n_cities)
        Returns:
            logits: torch tensor with shape (n_particles, 3), logits for (w, c1, c2)
        '''
        x = x.view(1, x.size(0), x.size(1))  # shape: (1, n_particles, n_cities)
        o = self.avgpool(self.conv1(x))
        o = self.avgpool(self.conv2(o))
        o = self.conv3(o)
        o = self.avg_global_pool(o)
        return self.fc(o).squeeze()

In [16]:
import torch

t = torch.randn(128, 50)

net = TSPVectorCNN1DModel(n_particles=128)
o = net(t)

In [20]:
net = nn.BatchNorm1d(50)
o = net(t)
o.shape

torch.Size([128, 50])